# Algorytmy Tekstowe 
---
## Zaawansowane scrapowanie - Web Scraping - część 2 z 2
- ### Praktyczny parser
- ### Iteratory, Generatory i `yield` 
- ### Zaawansowane scrapowanie przy użyciu biblioteki `Scrapy`
 - #### _"Grzeczne"_ pająki w Scrapy
 - #### Rozbudowany pająk do pobierania __*różnych*__ stron
---

### art. 8 ustawy z dnia 27 lipca 2001 r. o ochronie baz danych 

- #### 1. Wolno korzystać z istotnej, co do jakości lub ilości, części rozpowszechnionej bazy danych:
  - 1) ...
  - 2)   w charakterze ilustracji, w celach _**dydaktycznych lub badawczych**_ ze wskazaniem źródła, jeżeli takie korzystanie jest uzasadnione niekomercyjnym celem, dla którego wykorzystano bazę,
  

---
https://www.morizon.pl/

---

In [ ]:
# Jeśli logi Ci przeskadzają, odkomentuj to poniżej:

# import logging
# logger = logging.getLogger()
# logger.setLevel(logging.CRITICAL)

# Praktyczny parser
## Wyciągamy z ogłoszenia cenę mieszkania

In [1]:
import requests
from bs4 import BeautifulSoup


url='https://www.morizon.pl/oferta/sprzedaz-mieszkanie-warszawa-wlochy-plastyczna-31m2-mzn2046023842'
page = requests.get(url)
soup = BeautifulSoup(page.content, 'html.parser')

In [2]:
value = [flat for flat in soup.find_all('span', class_ = "details-price__item")][0]

In [3]:
value

<span class="details-price__item" data-cy="priceRowPrice" data-v-f63f87dc="">662 340 zł</span>

In [4]:
value.next_sibling

<span class="details-price__item details-price__item--m2" data-cy="detailsRowTextPriceM2Formatted" data-v-f63f87dc="">21 000 zł/m²</span>

In [5]:
def text_to_float(txt):
    return float(''.join(c for c in txt if c in [str(i) for i in range(10)]))

In [6]:
price = text_to_float(value.text)
price

662340.0

In [7]:
value.next_sibling

<span class="details-price__item details-price__item--m2" data-cy="detailsRowTextPriceM2Formatted" data-v-f63f87dc="">21 000 zł/m²</span>

In [8]:
price_per_meter = text_to_float(value.next_sibling.text)
price_per_meter

21000.0

## Więcej szczegółów

In [9]:
details_list = soup.find_all('div', class_ = "details-highlighted-parameters__item-value")

In [10]:
pokoje = details_list[0].text
pokoje

'1'

In [11]:
powierzchnia =  details_list[1].text
powierzchnia

'31,54 m²'

In [12]:
pietro = details_list[2].text
pietro

'1 z 4'

In [13]:
[flat for flat in soup.find_all('h1')][0].text

'Mieszkanie na sprzedaż, 31 m², 1 pokój Warszawa Włochy'

In [14]:
def text_to_float(txt):
    return float(''.join(c for c in txt if c in [str(i) for i in range(10)]))
    
def flat_to_dict(soup):
    value = [flat for flat in soup.find_all('span', class_ = "details-price__item")][0]
    price = text_to_float(value.text)
    details_list = soup.find_all('div', class_ = "details-highlighted-parameters__item-value")
    rooms = details_list[0].text
    area =  details_list[1].text
    floor = details_list[2].text
    title = [flat for flat in soup.find_all('h1')][0].text
    return {
        'price' : price,
        'area': area,
        'rooms':  rooms,
        'floor': floor,
        'title': title,
    }

In [15]:
flat_to_dict(soup)

{'price': 662340.0,
 'area': '31,54 m²',
 'rooms': '1',
 'floor': '1 z 4',
 'title': 'Mieszkanie na sprzedaż, 31 m², 1 pokój Warszawa Włochy'}

---
## Iteratory, Generatory i `yield` 

In [16]:
print (range(5))

range(0, 5)


In [17]:
for i in range(5):
    print (i)

0
1
2
3
4


In [18]:
mystr = "banana"
myit = iter(mystr)

print(next(myit))
print(next(myit))
print(next(myit))
print(next(myit))
print(next(myit))
print(next(myit))

b
a
n
a
n
a


In [19]:
print(next(myit)) # error

StopIteration: 

In [20]:
class Counter:
    def __init__(self, low, high):
        self.current = low - 1
        self.high = high

    def __iter__(self):
        return self

    def __next__(self): 
        self.current += 1
        if self.current < self.high:
            return self.current
        raise StopIteration


for c in Counter(3, 9):
    print(c)

3
4
5
6
7
8


### *Generatory* są mechanizmem
* tworzenia iteratorów
* Zwraca dane przez *yield*
* Każde wywołanie _next()_ zaczyna od miejsca gdzie skończył poprzedni krok
* _next()_ tworzona jest automatycznie


In [21]:
def reverse(data):
    for index in range(len(data)-1, -1, -1):
        print(index)
        yield data[index]

In [22]:
for c in reverse('Python'):
    print (c)
    print()

5
n

4
o

3
h

2
t

1
y

0
P



# Generatory, Yield

In [ ]:
mylist = [0, 1, 4]
for i in mylist:
    print(i)

In [ ]:
mylist = [x*x for x in range(3)]
for i in mylist:
    print(i)

In [ ]:
mylist = (x*x for x in range(3))
for i in mylist:
    print(i)

In [ ]:
def create_generator():
    mylist = range(3)
    for i in mylist:
        yield i*i
        
for i in create_generator():
    print(i)

---
# Zaawansowane scrapowanie przy użyciu biblioteki `Scrapy`

https://scrapy.org/

## _"Grzeczne"_ pająki w Scrapy


- 1. Po pierwsze - nie szkodzić! Nie obciążaj niepotrzebnie strony scrapowanej
- 2. Przestrzegaj `robots.txt` i warunków korzystania z usługi
- 5. Nie ukrywaj się

#### Z dokumentacji:
- Scrapy doesn’t wait a fixed amount of time between requests, but uses a random interval between `0.5 * DOWNLOAD_DELAY` and `1.5 * DOWNLOAD_DELAY`.
- `AUTOTHROTTLE_ENABLED` - download delay for next requests is set to the average of previous download delay and the target download delay;

### `FAIL2BAN` - typowe zabezpieczenia

Przykład z dokumentacji:

*As you can see in my example, I have set up 300 maxretry and 300 for findtime, so, we need to have 300 GETs from the same IP in a time window of 300 seconds to have the originating IP blocked.*


In [ ]:
import scrapy
import scrapy.crawler as crawler
from bs4 import BeautifulSoup

from scrapy.crawler import CrawlerProcess

class Spider1(scrapy.Spider):
    name = 'my_morizon_spider'
    start_urls = [
        'https://www.morizon.pl/mieszkania/warszawa/'
        ]
    
    item_urls2 = ['https://www.morizon.pl/mieszkania/warszawa/?page=2']    
           
    custom_settings = {
        'DOWNLOAD_DELAY': '2.0',
        'ROBOTSTXT_OBEY': True,
        'AUTOTHROTTLE_ENABLED': True,
        'USER_AGENT': 'My Morizon Demo Bot (korzycki@agh.edu.pl)'
    }

    top_url = 'https://www.morizon.pl/'
    
    
    def parse(self, response):
        self.logger.info('1. Got successful response from {}'.format(response.url))

        for item_url in self.item_urls2:
                yield scrapy.Request(item_url, self.parse)


In [ ]:
import scrapy
import scrapy.crawler as crawler
from bs4 import BeautifulSoup

from scrapy.crawler import CrawlerProcess

class MySpider(scrapy.Spider):
    name = 'my_morizon_spider'
    start_urls = [
        'https://www.morizon.pl/mieszkania/warszawa/'
        ]
    
    custom_settings = {
        'DOWNLOAD_DELAY': '4.0',
        'ROBOTSTXT_OBEY': True,
        'AUTOTHROTTLE_ENABLED': True,
        'USER_AGENT': 'My Morizon Demo Bot (michal.korzycki@gmail.com)'
    }

    
    def parse(self, response):
        self.logger.info('Got successful response from {}'.format(response.url))
        soup = BeautifulSoup(response.body, 'lxml')

        links = [link.get('href')
                for link in soup.find_all('a')]

        links= links[:100]
        print(links)
        
        for item_url in links:
            yield scrapy.Request(item_url, self.parse_item)
        
    def parse_item(self, response): #item_url - odwiedzanie strony, #self.parse_item - przetworzenie przy pomocy funkcji
        self.logger.info('Got successful response from {}'.format(response.url))

In [ ]:
process = CrawlerProcess()
process.crawl(MySpider)
process.start()

In [ ]:
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import scrapy
import scrapy.crawler as crawler
from bs4 import BeautifulSoup
from datetime import datetime 
from scrapy.crawler import CrawlerProcess


def text_to_float(txt):
    return ''.join(c for c in txt if c in [str(i) for i in range(10)])
    
def flat_to_dict(soup):
    node = [flat for flat in soup.find_all('div', id = "basic-info-price-row")][0]
    spans = node.find_all('span')
    no_spans = (len(spans)<2)
    next_data = node.next_sibling()
    locations = values = [flat for flat in soup.find_all('h2')][1]
    location = [n.text.strip().strip(',') for n in locations.find_all('span')]
    return {
        'price' : '' if no_spans else text_to_float(spans[0].text),
        'price_per_meter' : '' if no_spans else text_to_float(spans[1].text),
        'rooms':  next_data[0].span.text,
        'floor': next_data[6].span.text,
        'location': ", ".join(location),
        'title': [flat for flat in soup.find_all('h1')][0].text
    }
    

result = []

class MySpider(scrapy.Spider):
    name = 'my_morizon_spider'
    start_urls = [
        'https://www.morizon.pl/mieszkania/warszawa/?page=%d' %i  for i in range(1,3)
        ]
   
    custom_settings = {
        'DOWNLOAD_DELAY': '4.0',
        'ROBOTSTXT_OBEY': True,
        'AUTOTHROTTLE_ENABLED': True,
        'USER_AGENT': 'My Morizon Demo Bot (michal.korzycki@gmail.com)'
    }
    
    def __init__(self, *args, **kwargs):
        logger = logging.getLogger("scrapy.spidermiddlewares.httperror")
        logger.setLevel(logging.WARNING)
        logger = logging.getLogger("scrapy.core.engine")
        logger.setLevel(logging.WARNING)
        super().__init__(*args, **kwargs)

    def parse(self, response):
        self.logger.setLevel(logging.INFO)
        # self.logger.info('Got successful response from {}'.format(response.url))
        soup = BeautifulSoup(response.body, 'lxml')

        links = [link.get('href')
                for link in soup.find_all('a')]
        
        links = [ 'https://www.morizon.pl'+link for link in links if link.startswith('/oferta') ]
        
        for item_url in links:
            yield scrapy.Request(item_url, self.parse_item)
        
    def parse_item(self, response): #item_url - odwiedzanie strony, #self.parse_item - przetworzenie przy pomocy funkcji
        self.logger.info('Got successful response from {}'.format(response.url))
        soup = BeautifulSoup(response.body, 'lxml')
        data = flat_to_dict(soup)
        data['url'] = response.url
        result.append(data)
        
    def spider_closed(self, spider):
        try:
            spider.logger.info('Spider closed: %s', spider.name)
            now = datetime.now().strftime("%Y-%m-%d")
            spider.logger.info('Spider closed: %s', spider.name)
            df = pd.Dataframe(result)
            fname = f"data/morizon-{now}.csv"
            print(fname)
            spider.logger.info('Spider writing to: %s', fname)
            df.to_csv(fname)
        except Exception as e:
            print(e)

In [ ]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

process = CrawlerProcess()
process.crawl(MySpider)
process.start()

In [ ]:
import pandas as pd

df = pd.DataFrame.from_records(result)

In [ ]:
df.columns

In [ ]:
df.sample(15)

In [ ]:
pd.DataFrame(df.iloc[31])

In [ ]:
from datetime import datetime

now = datetime.now().strftime("%Y-%m-%d")
fname = f"data/morizon-{now}.csv"
fname

In [ ]:
df.to_csv(fname)